# RAG (Retrieval-Augmented Generation) — Interview Q&A

> **Target roles:** Research Scientist · ML Engineer · Foundational Modeling  
> Covers Naive RAG, Advanced RAG, Modular RAG, GraphRAG, HyDE, and more.

## 1. RAG Fundamentals

**Q1. What is RAG and why was it introduced?**  
RAG (Lewis et al. 2020) augments LLMs with a retrieval mechanism that fetches relevant documents from an external knowledge base at inference time. Motivation: LLMs have static parametric knowledge (frozen at training), can hallucinate, and can't access private/recent data. RAG addresses all three: retrieval provides dynamic, verifiable, domain-specific context.

**Q2. What are the three main components of a RAG system?**  
1. **Retriever:** Encodes queries and documents into vector embeddings; retrieves top-k relevant chunks via similarity search (usually cosine or dot product).  
2. **Knowledge Base:** Chunked, embedded, and indexed documents stored in a vector database (Pinecone, Weaviate, FAISS, Chroma).  
3. **Generator:** LLM that conditions on retrieved context + user query to produce a grounded answer.

**Q3. What is the basic RAG pipeline?**  
**Indexing (offline):** Load documents → chunk → embed → store in vector DB.  
**Retrieval (online):** Embed query → ANN search → return top-k chunks.  
**Generation:** Concatenate [retrieved chunks + query] as LLM context → generate answer.

**Q4. What is chunking and why does it matter?**  
Splitting documents into smaller segments before embedding. Chunk size trade-off: too small = loses context; too large = embeddings diluted, exceeds LLM context window. Common strategies: fixed-size (512 tokens with overlap), sentence-based, semantic chunking (split on semantic similarity boundaries), recursive character splitting. Overlap (e.g., 50 tokens) prevents splitting important context across chunks.


## 2. Naive RAG

**Q5. What is Naive RAG and what are its limitations?**  
The basic pipeline: chunk → embed → retrieve → generate. Works as a baseline but has several failure modes:

**Retrieval failures:**  
- Low precision: retrieved chunks irrelevant (noisy context confuses LLM)  
- Low recall: relevant information missed (wrong embedding model, suboptimal chunking)  
- Semantic mismatch: query phrasing differs from document phrasing  

**Generation failures:**  
- Context window overflow with many chunks  
- Ignoring retrieved context (LLM relies on parametric knowledge anyway)  
- Hallucination even with correct context  
- Ordering effects: "lost-in-the-middle" problem (LLMs attend poorly to middle of long contexts)

**Q6. What is the lost-in-the-middle problem?**  
Liu et al. (2023): LLMs best attend to context at the beginning and end of the prompt; relevant information in the middle is often ignored. Mitigation: rerank retrieved chunks and place most relevant first/last, limit number of chunks, or use models with improved long-context attention.


## 3. Advanced RAG

**Q7. What is Advanced RAG and what improvements does it add?**  
Advanced RAG addresses Naive RAG's limitations with improvements at every stage:

**Pre-retrieval:** Query transformation, routing, better chunking  
**Retrieval:** Hybrid search (dense + sparse), reranking, parent-child document retrieval  
**Post-retrieval:** Context compression, reranking, filtering  
**Generation:** Iterative refinement, citation grounding

**Q8. What is HyDE (Hypothetical Document Embeddings)?**  
Gao et al. (2022): Instead of embedding the user query directly, use an LLM to *hallucinate* a hypothetical answer/document, then embed that hypothetical document for retrieval. Rationale: hypothetical answers share vocabulary and phrasing with real documents better than short queries. Works especially well when queries are short/vague.

**Q9. What is query decomposition / step-back prompting?**  
**Query decomposition:** Break complex questions into simpler sub-queries, retrieve for each, combine results.  
**Step-back prompting:** Generate a higher-level, more abstract question first (e.g., "What is the general principle?"), retrieve on that, then answer the specific question. Improves recall for narrow/specific queries.

**Q10. What is hybrid search in RAG?**  
Combines dense retrieval (embedding-based, captures semantic similarity) with sparse retrieval (BM25/TF-IDF, exact keyword matching). Scores are fused with Reciprocal Rank Fusion (RRF) or linear combination. Critical for: domain-specific terminology, product codes, names, abbreviations where semantic embeddings fail. Best of both worlds.

**Q11. What is a reranker and when is it used?**  
Two-stage retrieval: (1) fast approximate retrieval of top-100 candidates via ANN search; (2) cross-encoder reranker scores each (query, document) pair jointly and re-orders. Cross-encoders are more accurate (full query-document interaction) but slow (O(k) LLM calls). Models: BGE-Reranker, Cohere Rerank, ColBERT. Always improves precision at cost of latency.

**Q12. What is Parent Document Retrieval?**  
Index smaller child chunks for precise retrieval; when a child chunk is retrieved, return its parent (larger context). Combines retrieval precision of small chunks with generation quality of large context. Implemented in LangChain's ParentDocumentRetriever.


## 4. Modular RAG & Advanced Architectures

**Q13. What is Modular RAG?**  
Gao et al. (2023): Extends Advanced RAG into composable modules that can be recombined into custom pipelines. Modules: Search, Memory, Fusion, Routing, Predict, Task Adapter. Allows flexible, task-specific RAG architectures beyond fixed retrieve-then-generate pipeline.

**Q14. What is Self-RAG?**  
Asai et al. (2023): LLM learns to (1) decide *whether* to retrieve at all, (2) evaluate quality of retrieved passages, (3) assess whether its own output is supported by the context. Introduces special tokens: [Retrieve], [IsRel], [IsSup], [IsUse]. More autonomous — selectively retrieves only when needed. Reduces unnecessary retrieval noise.

**Q15. What is FLARE (Forward-Looking Active Retrieval)?**  
Iterative retrieval: LLM generates one sentence at a time. When the model is uncertain (low token probability), it pauses, uses the upcoming predicted text as a query to retrieve relevant information, then continues generation with that context. Enables multi-hop reasoning and real-time knowledge grounding.

**Q16. What is Corrective RAG (CRAG)?**  
Yan et al. (2024): Adds a retrieval evaluator that assesses the quality of retrieved documents. If documents are deemed irrelevant, the system falls back to web search or uses alternative sources. Corrective mechanism makes RAG more robust to retrieval failures.

**Q17. What is GraphRAG?**  
Edge et al. (2024, Microsoft): Builds a knowledge graph from source documents (entity extraction + relationship extraction). At query time, traverses the graph to find related information, supporting multi-hop reasoning that vector search misses. Especially powerful for: "What are all the relationships between X and Y across the entire corpus?" queries. More expensive to build but handles complex relational queries.

**Q18. What is Agentic RAG?**  
RAG within an agent framework where retrieval is one tool among many. The agent decides when to retrieve, what to retrieve, how to decompose the problem, and can iteratively search (ReAct pattern). Enables complex multi-step queries. Examples: LangGraph RAG agents, AutoGPT with retrieval.


## 5. Evaluation & Production

**Q19. How do you evaluate a RAG system?**  
**Retrieval metrics:**  
- Context Precision: fraction of retrieved chunks that are relevant  
- Context Recall: fraction of relevant information that was retrieved  

**Generation metrics:**  
- Faithfulness: is the answer supported by the context? (no hallucination)  
- Answer Relevance: does the answer address the question?  
- Answer Correctness: is the answer factually correct?  

**Frameworks:** RAGAS (automated RAG evaluation), TruLens, DeepEval. Often requires LLM-as-judge for semantic evaluation.

**Q20. What is the embedding model choice for RAG?**  
Key factors: (1) domain fit — general (OpenAI text-embedding-3, BGE) vs domain-specific; (2) dimension — higher is more expressive but slower; (3) context length — some embeddings truncate at 512 tokens; (4) instruction-following — E5/GTE/BGE-Instruct use task prefixes. MTEB benchmark is the standard evaluation. Don't use the same model for documents and reranking.

**Q21. What are the key trade-offs in RAG system design?**  
| Parameter | Low | High |
|---|---|---|
| Chunk size | High precision, low context | Low precision, rich context |
| Top-k retrieved | Miss info, fast | Noisy context, slow, expensive |
| Reranker | Fast, lower precision | Slow, higher precision |
| Embedding dim | Fast, less expressive | Slow, more expressive |
| Overlap | Duplicate info | Better boundary handling |


## 6. ⚠️ Trick Questions

**Q22. ⚠️ Does more retrieved context always improve generation quality?**  
No. The lost-in-the-middle problem means LLMs often ignore middle context. Too many chunks can dilute the truly relevant information, causing the LLM to generate worse answers. Optimal k is task-dependent — empirically tune it. Quality beats quantity.

**Q23. ⚠️ Can RAG fully replace fine-tuning?**  
No. They complement each other. RAG is better for: factual knowledge, private data, recent information, citation grounding. Fine-tuning is better for: style/tone adaptation, task-specific formatting, domain-specific reasoning patterns, low-latency (no retrieval step). RAG + fine-tuning is often optimal.

**Q24. ⚠️ Is vector similarity always the right retrieval metric?**  
No. Cosine similarity on dense embeddings works well for semantic matching but fails for: exact keyword matching (product IDs, names), Boolean filters, structured data queries. Hybrid search (dense + BM25) addresses this. For structured data, SQL-based retrieval may be more appropriate.

**Q25. ⚠️ What happens when retrieved context contradicts parametric knowledge?**  
LLMs tend to prioritize parametric knowledge for highly confident facts. This means RAG may not override strongly held LLM beliefs — a critical failure mode. Solutions: explicit prompting ("always use the provided context"), fact-checking, or CRAG-style correction mechanisms.

**Q26. ⚠️ Does a longer context window eliminate the need for RAG?**  
Not really. Even with 1M token context windows: (1) latency and cost scale with context length; (2) attention quality degrades on very long contexts; (3) not all documents fit even in 1M tokens; (4) retrieval is still more efficient. Context windows are complementary, not a replacement.


## 7. Quick Reference Summary

### RAG Evolution Timeline
```
Naive RAG → Advanced RAG → Modular RAG → Agentic RAG
  (2020)       (2022-23)       (2023)        (2024)
```

### Which RAG to Use?
| Scenario | Recommended Approach |
|---|---|
| Simple Q&A on documents | Naive RAG or Advanced RAG |
| Complex multi-hop reasoning | GraphRAG or FLARE |
| Vague/short queries | HyDE |
| High precision required | Advanced RAG + Reranker |
| Autonomous agent | Agentic RAG |
| Uncertain retrieval quality | Self-RAG or CRAG |
| Mixed exact/semantic queries | Hybrid Search |

### Production Checklist
- [ ] Chunking strategy tuned for document type  
- [ ] Embedding model benchmarked on your domain (MTEB)  
- [ ] Hybrid search enabled (dense + BM25)  
- [ ] Cross-encoder reranker for precision  
- [ ] RAGAS evaluation baseline established  
- [ ] Prompt instructs LLM to use provided context  
- [ ] Hallucination detection / citation grounding  
- [ ] Retrieval latency profiled and cached if needed


In [ ]:
# RAG System Skeleton (illustrative - shows key components)
# This demonstrates the architecture without requiring API keys

from dataclasses import dataclass
from typing import List
import numpy as np

@dataclass
class Document:
    content: str
    doc_id: str
    
class SimpleRAG:
    """Minimal RAG illustration. Production would use FAISS/Chroma/Pinecone."""
    
    def __init__(self, embedding_dim: int = 64):
        self.embedding_dim = embedding_dim
        self.documents: List[Document] = []
        self.embeddings: List[np.ndarray] = []
    
    def _mock_embed(self, text: str) -> np.ndarray:
        """In production: use sentence-transformers, OpenAI embeddings, etc."""
        np.random.seed(hash(text[:50]) % (2**31))
        return np.random.randn(self.embedding_dim)
    
    def add_documents(self, docs: List[Document]):
        for doc in docs:
            self.documents.append(doc)
            self.embeddings.append(self._mock_embed(doc.content))
        print(f"Indexed {len(docs)} documents. Total: {len(self.documents)}")
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Document]:
        if not self.documents:
            return []
        query_emb = self._mock_embed(query)
        # Cosine similarity
        norms = np.linalg.norm(self.embeddings, axis=1)
        scores = np.dot(self.embeddings, query_emb) / (norms * np.linalg.norm(query_emb) + 1e-8)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.documents[i], float(scores[i])) for i in top_indices]
    
    def generate(self, query: str, retrieved_docs: list) -> str:
        """In production: pass context + query to LLM (GPT-4, Claude, Llama)."""
        context = "\n\n".join([f"[Doc {i+1}]: {doc.content[:200]}" 
                                for i, (doc, score) in enumerate(retrieved_docs)])
        prompt = f"""Context:\n{context}\n\nQuestion: {query}\n\nAnswer based on the context above."""
        return f"[LLM would answer here given prompt of {len(prompt)} chars]"

# Example usage
rag = SimpleRAG()
docs = [
    Document("Transformer models use self-attention to process sequences in parallel.", "doc1"),
    Document("BERT is a bidirectional transformer pre-trained on masked language modeling.", "doc2"),
    Document("GPT uses causal attention for autoregressive text generation.", "doc3"),
    Document("RAG combines retrieval with generation for knowledge-grounded responses.", "doc4"),
]
rag.add_documents(docs)
results = rag.retrieve("How does RAG work?", top_k=2)
for doc, score in results:
    print(f"Score: {score:.3f} | {doc.content[:80]}...")
